In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

In [40]:
df= pd.read_csv("../data/german_credit_data.csv")

In [41]:
df.head()   

,Checking_Account_Status,Duration,Credit_History,Purpose,Credit_Amount,Savings_Account,Employment_Duration,Installment_Rate,Personal_Status_Sex,Other_Debtors,...,Property,Age,Other_Installment_Plans,Housing,Existing_Credits,Job,Num_Dependents,Telephone,Foreign_Worker,Target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,0
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,1
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,0
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,0
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,1


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Checking_Account_Status  1000 non-null   object
 1   Duration                 1000 non-null   int64 
 2   Credit_History           1000 non-null   object
 3   Purpose                  1000 non-null   object
 4   Credit_Amount            1000 non-null   int64 
 5   Savings_Account          1000 non-null   object
 6   Employment_Duration      1000 non-null   object
 7   Installment_Rate         1000 non-null   int64 
 8   Personal_Status_Sex      1000 non-null   object
 9   Other_Debtors            1000 non-null   object
 10  Residence_Duration       1000 non-null   int64 
 11  Property                 1000 non-null   object
 12  Age                      1000 non-null   int64 
 13  Other_Installment_Plans  1000 non-null   object
 14  Housing                  1000 non-null   

In [43]:
X = df.drop(columns=["Target"])
y = df["Target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [44]:
cat_cols= X.select_dtypes(include=['object']).columns.tolist()
robust=["Duration","Age","Credit_Amount"]
mixman=["Installment_Rate","Residence_Duration","Existing_Credits","Num_Dependents"]

In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler
preprocessor= ColumnTransformer(
    transformers=[
        ("robust", RobustScaler(), robust),
        ("mixman", MinMaxScaler(), mixman),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ])
cv= StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [46]:
params= {
    "C": 0.10,
    "penalty": "l2",
    "solver": "saga",
    "max_iter": 1000,
    "random_state": 42,
    "class_weight": "balanced",

}


Logistic_Regresion= Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(**params))
])

lr_scores= cross_val_score(Logistic_Regresion, X_train, y_train, cv=cv, scoring="roc_auc")

print(f"Scores for each fold: {lr_scores}")
print(f"AUC: {np.mean(lr_scores):.4f} ± {np.std(lr_scores):.4f}")

Scores for each fold: [0.82849702 0.82254464 0.78385417 0.68991815 0.8061756 ]
AUC: 0.7862 ± 0.0506


In [59]:
SVC= Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(random_state=42, probability=True, class_weight="balanced"))
])

svc_scores= cross_val_score(SVC, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"Scores for each fold: {svc_scores}")
print(f"AUC: {np.mean(svc_scores):.4f} ± {np.std(svc_scores):.4f}")

Scores for each fold: [0.80375744 0.78999256 0.79854911 0.70926339 0.81863839]
AUC: 0.7840 ± 0.0385


In [48]:
XGB= Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        max_depth=4,
        n_estimators=1000,
        learning_rate=0.01,
        eval_metric="auc",
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
    )),
])
xgb_scores= cross_val_score(XGB, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"Scores for each fold: {xgb_scores}")
print(f"AUC: {np.mean(xgb_scores):.4f} ± {np.std(xgb_scores):.4f}")

Scores for each fold: [0.84077381 0.79947917 0.80375744 0.72488839 0.82049851]
AUC: 0.7979 ± 0.0393


In [49]:
RandomForest= Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    ))
])
rf_scores= cross_val_score(RandomForest, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"Scores for each fold: {rf_scores}")
print(f"AUC: {np.mean(rf_scores):.4f} ± {np.std(rf_scores):.4f}")

Scores for each fold: [0.82300967 0.82375372 0.76943824 0.71605283 0.81687128]
AUC: 0.7898 ± 0.0421


In [50]:
DecisionTreeClassifier= Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced",
    ))
])
dt_scores= cross_val_score(DecisionTreeClassifier, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"Scores for each fold: {dt_scores}")
print(f"AUC: {np.mean(dt_scores):.4f} ± {np.std(dt_scores):.4f}")

Scores for each fold: [0.66220238 0.61458333 0.5922619  0.58928571 0.64434524]
AUC: 0.6205 ± 0.0287


In [51]:
cv_summary = pd.DataFrame([
    {"model": "Logistic Regression", "cv_auc_mean": np.mean(lr_scores), "cv_auc_std": np.std(lr_scores)},
    {"model": "SVC", "cv_auc_mean": np.mean(svc_scores), "cv_auc_std": np.std(svc_scores)},
    {"model": "XGBoost", "cv_auc_mean": np.mean(xgb_scores), "cv_auc_std": np.std(xgb_scores)},
    {"model": "Random Forest", "cv_auc_mean": np.mean(rf_scores), "cv_auc_std": np.std(rf_scores)},
    {"model": "Decision Tree", "cv_auc_mean": np.mean(dt_scores), "cv_auc_std": np.std(dt_scores)},
])
cv_summary.sort_values("cv_auc_mean", ascending=False)

,model,cv_auc_mean,cv_auc_std
2,XGBoost,0.797879,0.039284
1,SVC,0.792374,0.039735
3,Random Forest,0.789825,0.042052
0,Logistic Regression,0.786198,0.050566
4,Decision Tree,0.620536,0.028686


In [60]:
models = {
    "Logistic Regression": Logistic_Regresion,
    "SVC": SVC,
    "XGBoost": XGB,
    "Random Forest": RandomForest,
    "Decision Tree": DecisionTreeClassifier,
}

test_results = []
for name, estimator in models.items():
    fitted = estimator.fit(X_train, y_train)
    proba = fitted.predict_proba(X_test)[:, 1]
    preds = fitted.predict(X_test)
    test_results.append({
        "model": name,
        "test_auc": roc_auc_score(y_test, proba),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, preds),
        "test_accuracy": accuracy_score(y_test, preds),
    })

test_results = pd.DataFrame(test_results).sort_values("test_auc", ascending=False)
test_results

,model,test_auc,test_balanced_accuracy,test_accuracy
1,SVC,0.814881,0.757143,0.760
0,Logistic Regression,0.808214,0.741667,0.725
2,XGBoost,0.796667,0.691667,0.775
3,Random Forest,0.796071,0.627381,0.745
4,Decision Tree,0.600000,0.600000,0.660


In [53]:
from optbinning import BinningProcess

In [54]:
selection_criteria = {
    "iv": {"min": 0.02}
}

# 3. Definición del BinningProcess
# Este objeto es el corazón del pipeline: hace el binning y el WoE
binning_process = BinningProcess(
    variable_names=X.columns.tolist(),
    categorical_variables=cat_cols,
    selection_criteria=selection_criteria
)

# 4. Construcción del Pipeline
# El Pipeline aplica el BinningProcess y luego la Regresión Logística
scoring_pipeline = Pipeline([
    ('binning_process', binning_process),
    ('classifier', LogisticRegression(
        class_weight='balanced', # Crucial para datos desbalanceados
        solver='lbfgs',
        max_iter=1000,
        penalty='l2',
        C=1,
    ))
])


In [55]:
woe_scores= cross_val_score(scoring_pipeline, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"Scores for each fold: {woe_scores}")
print(f"AUC: {np.mean(woe_scores):.4f} ± {np.std(woe_scores):.4f}")

Scores for each fold: [0.81026786 0.81733631 0.81361607 0.70851935 0.79613095]
AUC: 0.7892 ± 0.0410


In [56]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [61]:
# 1. Volvemos a importar para recuperar las clases originales
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 2. Definimos el pipeline con un nombre de variable que NO sea SVC
pipeline_svc_woe = Pipeline(steps=[
    ('binning_process', binning_process),
    ('classifier', SVC(probability=True,kernel="linear",class_weight="balanced", random_state=42))
])

# 3. Probamos la ejecución
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipeline_svc_woe, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"AUC con SVC + WoE: {scores.mean():.4f}")
print(f"Scores for each fold: {scores}")
print(f"AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

AUC con SVC + WoE: 0.7901
Scores for each fold: [0.81287202 0.81826637 0.81156994 0.71000744 0.79780506]
AUC: 0.7901 ± 0.0406


In [63]:
pipeline_random_forest_woe = Pipeline(steps=[
    ('binning_process', binning_process),
    ('classifier', RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    ))
])

scores = cross_val_score(pipeline_random_forest_woe, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"AUC con Random Forest + WoE: {scores.mean():.4f}")
print(f"Scores for each fold: {scores}")
print(f"AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

AUC con Random Forest + WoE: 0.7617
Scores for each fold: [0.77483259 0.7859003  0.77074033 0.71465774 0.76218378]
AUC: 0.7617 ± 0.0247


In [64]:
pipeline_xgb_woe = Pipeline(steps=[
    ('binning_process', binning_process),
    ('classifier', XGBClassifier(
        max_depth=4,
        n_estimators=1000,
        learning_rate=0.01,
        eval_metric="auc",
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
    )),
])
scores = cross_val_score(pipeline_xgb_woe, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"AUC con Random Forest + WoE: {scores.mean():.4f}")
print(f"Scores for each fold: {scores}")
print(f"AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

AUC con Random Forest + WoE: 0.7801
Scores for each fold: [0.79910714 0.81045387 0.80022321 0.70889137 0.78199405]
AUC: 0.7801 ± 0.0368
